# Quill — Notebook 07: String Predicates

**Goal:** build a reversible quantum circuit that checks a condition on a `Cord`.

Our example asks: **“Is this two-character Cord exactly `DM`?”** For a two-character Cord, that is equivalent to “starts with `D` and ends with `M`.” We will compute the answer into an extra qubit called a *flag*.

This is a reversible circuit, not yet a quantum speedup or a search algorithm. We are learning how to express a predicate as quantum gates.

## 1. From a question to a Boolean predicate

Let the input be a two-character Cord, encoded as 16 qubits. Define

$$P(s)=\begin{cases}1 & \text{if }s=\text{“DM”},\\0 & \text{otherwise.}\end{cases}$$

The circuit should implement

$$|s\rangle|0\rangle_f \longmapsto |s\rangle|P(s)\rangle_f.$$

The input is preserved and the flag stores the answer. This is reversible because the input is not erased.

## 2. Why ASCII makes this a bit-level test

`D` has ASCII value 68 (`01000100`) and `M` has value 77 (`01001101`). The circuit checks all 16 encoded bits against those expected values.

A multi-controlled X (`MCX`) flips the flag only when every control qubit is 1. For each expected zero bit, we temporarily apply X to turn that zero into a one; after the check, we undo those X gates. This “mask, check, unmask” pattern leaves the input unchanged.

In [ ]:
# Google Colab: install if needed
%pip -q install qiskit qiskit-aer

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector

sim = AerSimulator()

## 3. Build the reversible predicate circuit

We use qubits `0–7` for the first character, `8–15` for the second, and qubit `16` for the flag. Within each byte, qubit 0 is the least-significant bit (LSB), even though binary strings are usually written most-significant bit first.

In [ ]:
def cord_bits(text):
    """Return ASCII bits in Qiskit's little-endian qubit order per character."""
    bits = []
    for char in text:
        value = ord(char)
        bits.extend((value >> bit) & 1 for bit in range(8))
    return bits


def predicate_circuit(text, target="DM", measure=True):
    if len(text) != 2 or len(target) != 2:
        raise ValueError("This notebook's predicate uses exactly two characters.")

    expected = cord_bits(target)
    actual = cord_bits(text)
    qc = QuantumCircuit(17, 1 if measure else 0)

    # Prepare the classical input Cord in its ASCII basis state.
    for q, bit in enumerate(actual):
        if bit:
            qc.x(q)

    # Turn expected zero-bits into ones, so all 16 controls can be positive controls.
    zero_positions = [q for q, bit in enumerate(expected) if bit == 0]
    for q in zero_positions:
        qc.x(q)

    # Flag flips iff all 16 encoded bits match the target "DM".
    qc.mcx(list(range(16)), 16)

    # Undo the mask; the original Cord is restored.
    for q in reversed(zero_positions):
        qc.x(q)

    if measure:
        qc.measure(16, 0)
    return qc

qc = predicate_circuit("DM")
qc.draw("mpl")

## 4. Test a few Cords

The flag should be 1 only for `DM`. The other 16 qubits remain the encoded input.

In [ ]:
def run_predicate(text):
    qc = predicate_circuit(text)
    result = sim.run(qc, shots=100).result()
    counts = result.get_counts()
    return counts

for candidate in ["DM", "DA", "XM", "HI"]:
    print(candidate, "→", run_predicate(candidate))

## 5. Derive the action with state notation

For the matching input, after preparation the state is $|DM\rangle|0\rangle_f$. The predicate flips the flag:

$$|DM\rangle|0\rangle_f \longrightarrow |DM\rangle|1\rangle_f.$$

For a non-match such as `DA`, the control condition is not satisfied:

$$|DA\rangle|0\rangle_f \longrightarrow |DA\rangle|0\rangle_f.$$

In general, the reversible operation is

$$U_P:|s\rangle|b\rangle_f\longmapsto |s\rangle|b\oplus P(s)\rangle_f,$$

where $\oplus$ is XOR. Writing the answer into a flag instead of simply erasing or overwriting the input is what makes the operation reversible.

## 6. Check the circuit structure

Look at the circuit and identify three stages:

1. **Prepare** the input Cord.
2. **Mask → MCX → unmask** to compute the predicate without changing the input.
3. **Measure the flag** to read the answer.

This example checks one fixed string. It does **not** search a database of strings, and it does not provide quantum speedup. A later notebook can explore phase oracles and how predicates fit into quantum algorithms.

In [ ]:
# Inspect the circuit without measurement
predicate_circuit("DM", measure=False).draw("mpl")

## Exercises

1. Change the target to `HI` and test `HI`, `HO`, and `AI`.
2. Modify the function so it accepts a target argument throughout the code.
3. Explain why the circuit must preserve the input Cord.
4. Why does checking a single classical input not demonstrate quantum search?

**Next concept:** use a reversible predicate as a building block for a phase oracle—carefully distinguishing a genuine oracle from a classical pre-check.